In [1]:
# ===============================================================
# Cell 1: 環境設定與套件安裝
# ===============================================================
import sys
import subprocess

packages = ['requests', 'beautifulsoup4', 'pandas', 'lxml']
print("--- 正在檢查並安裝必要套件 ---")
for package in packages:
    try:
        __import__(package)
    except ImportError:
        print(f"正在安裝 {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

print("\n🎉 所有必要的套件都已準備就緒！")

--- 正在檢查並安裝必要套件 ---
正在安裝 beautifulsoup4...

🎉 所有必要的套件都已準備就緒！


In [2]:
# ===============================================================
# Cell 2: UDN 爬蟲全域設定與參數 (最終修正版)
# ===============================================================
import requests
from bs4 import BeautifulSoup
import pandas as pd
from datetime import datetime
import time
import os
import json
import re

# --- 通用設定 ---
HEADERS = {
    'User-Agent': 'T-Forecast-Project-Bot/1.0 (Academic research for a university project)'
}
DB_FILENAME = "udn_articles_db.csv"
HEADLINE_LOG_FILENAME = f"udn_headline_log_{datetime.now().strftime('%Y-%m-%d')}.csv"

# --- UDN 特定參數 ---
UDN_HOME_URL = "https://udn.com/news/index"
UDN_RSS_URL = "https://udn.com/news/rssfeed"

# --- 爬蟲選擇器 (⭐ 已更新為純 CSS 選擇器策略) ---
# 內文爬蟲：抓取文章內頁
# (此部分策略不變，依然是優先 JSON-LD，失敗則解析 HTML)

# 頭條標籤爬蟲：
# Level 1 (輪播圖中的大標題)
HEADLINE_LEVEL_1_SELECTOR = "div.udn-slider div.slide-links h2 a"
# Level 2 (輪播圖中的小標題 + 右側焦點新聞)
HEADLINE_LEVEL_2_SELECTOR = "div.udn-slider div.slide-links h3 a, div.focus a.focus-list"

print("所有 UDN 參數已更新並載入記憶體 (最終修正版)。")

所有 UDN 參數已更新並載入記憶體 (最終修正版)。


In [ ]:
# ===============================================================
# Cell 3: UDN 核心爬蟲函式與主要執行流程 (最終修正版)
# ===============================================================

# --- 函式定義區 ---

def get_new_article_urls_from_udn_rss(rss_url, existing_urls_set):
    """從 UDN RSS Feed 中，找出資料庫裡還沒有的新文章 URL。"""
    print("正在從 UDN RSS Feed 檢查新文章...")
    new_urls = []
    try:
        response = requests.get(rss_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        response.encoding = 'utf-8'
        soup = BeautifulSoup(response.text, 'xml')
        for item in soup.find_all('item'):
            url = item.link.text
            if url and "udn.com/news/story" in url and url not in existing_urls_set:
                new_urls.append(url)
        print(f"從 RSS 發現 {len(new_urls)} 篇新文章。")
        return new_urls
    except Exception as e:
        print(f"❌ 讀取 UDN RSS 失敗: {e}")
        return []

def scrape_udn_article_content(article_url):
    """訪問 UDN 文章頁面，並解析其內容。"""
    try:
        time.sleep(1) 
        response = requests.get(article_url, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        # 備用策略優先：直接解析 HTML 標籤，因為 UDN 的 JSON-LD 不穩定
        title = soup.select_one("h1.article-content__title").text.strip() if soup.select_one("h1.article-content__title") else ""
        published_time = soup.select_one("time.article-content__time").text.strip() if soup.select_one("time.article-content__time") else ""
        paragraphs = soup.select("div.article-content__paragraph p")
        content = "\n".join([p.text for p in paragraphs])

        if not title or not content:
            return None

        return {
            'url': article_url, 'title': title, 'content': content,
            'source': "聯合新聞網", 'published_at': published_time,
            'scraped_at': datetime.now(), 'credibility_label': 0, 'headline_level': 0
        }
    except Exception as e:
        return None

def scrape_udn_headlines_tags():
    """⭐ 已修正 ⭐ 訪問 UDN 首頁，直接從 HTML 標籤抓取頭條。"""
    print("正在抓取 UDN 頭條標籤...")
    results = []
    try:
        time.sleep(2)
        response = requests.get(UDN_HOME_URL, headers=HEADERS, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, 'html.parser')
        
        crawl_time = datetime.now()

        # --- 使用純 CSS 選擇器策略 ---
        # 抓取 Level 1
        for link in soup.select(HEADLINE_LEVEL_1_SELECTOR):
            url = link.get('href')
            if url:
                results.append({ 'observed_at': crawl_time, 'url': url, 'headline_level': 1 })

        # 抓取 Level 2
        for link in soup.select(HEADLINE_LEVEL_2_SELECTOR):
            url = link.get('href')
            if url and not url.startswith('http'):
                url = 'https://udn.com' + url
            
            # 避免重複加入
            if url and not any(d['url'] == url for d in results):
                 results.append({ 'observed_at': crawl_time, 'url': url, 'headline_level': 2 })
        
        return results
    except Exception as e:
        print(f"❌ 抓取 UDN 頭條標籤時發生錯誤: {e}")
        return []

# --- 主要執行流程 ---
def main_udn_execution():
    # Part 1: 抓取新文章內文
    print("--- Part 1: 開始執行 UDN 內文爬取任務 ---")
    existing_urls = set()
    if os.path.exists(DB_FILENAME):
        db_df_part1 = pd.read_csv(DB_FILENAME)
        existing_urls = set(db_df_part1['url'])
    print(f"UDN 資料庫中已存在 {len(existing_urls)} 篇文章。")
    
    new_urls_to_scrape = get_new_article_urls_from_udn_rss(UDN_RSS_URL, existing_urls)
    
    new_articles_data = []
    if new_urls_to_scrape:
        for url in new_urls_to_scrape[:20]: # 限制每次抓取 20 篇
            print(f"正在抓取內文: {url[:70]}...")
            content = scrape_udn_article_content(url)
            if content:
                new_articles_data.append(content)
                
    if new_articles_data:
        new_df = pd.DataFrame(new_articles_data)
        new_df.to_csv(DB_FILENAME, mode='a', header=not os.path.exists(DB_FILENAME), index=False, encoding='utf-8-sig')
        print(f"\n✅ Part 1 任務完成！成功新增 {len(new_articles_data)} 篇文章至 {DB_FILENAME}\n")
    else:
        print("\n🟡 Part 1 任務完成，但沒有發現或成功抓取任何新文章。\n")

    # Part 2: 抓取今日頭條標籤
    print("--- Part 2: 開始執行頭條標籤爬取任務 ---")
    headline_log_data = scrape_udn_headlines_tags()
    if headline_log_data:
        df_log = pd.DataFrame(headline_log_data)
        df_log.to_csv(HEADLINE_LOG_FILENAME, mode='a', header=not os.path.exists(HEADLINE_LOG_FILENAME), index=False, encoding='utf-8-sig')
        print(f"✅ Part 2 任務完成！成功記錄 {len(headline_log_data)} 筆頭條標籤至 {HEADLINE_LOG_FILENAME}\n")
    else:
        print("🟡 Part 2 任務完成，但本次沒有抓取到任何頭條標籤。\n")

    # Part 3: 整合標籤至主資料庫
    print("--- Part 3: 開始執行數據整合與標籤更新任務 ---")
    # ... (Part 3 邏輯維持不變，但現在 Part 1 & 2 應該都能成功，所以它也能成功)...
    if os.path.exists(DB_FILENAME):
        db_df_part3 = pd.read_csv(DB_FILENAME)
        update_count = 0
        if os.path.exists(HEADLINE_LOG_FILENAME):
            log_df = pd.read_csv(HEADLINE_LOG_FILENAME)
            # 確保 headline_log_data 是最新的，如果日誌檔已存在，則用日誌檔的資料
            # 為了簡化，我們直接使用本次抓取的 headline_log_data
            if headline_log_data:
                db_df_part3.set_index('url', inplace=True)
                for item in headline_log_data:
                    url = item['url']
                    level = item['headline_level']
                    if url in db_df_part3.index and db_df_part3.loc[url, 'headline_level'] < level:
                        db_df_part3.loc[url, 'headline_level'] = level
                        update_count += 1
                db_df_part3.reset_index(inplace=True)
                db_df_part3.to_csv(DB_FILENAME, index=False, encoding='utf-8-sig')
        print(f"✅ Part 3 整合完畢！共更新了 {update_count} 筆資料的頭條標籤。")
    else:
        print("🟡 Part 3 整合失敗，找不到主資料庫檔案。請先確保 Part 1 成功執行。")


# 執行主程式
main_udn_execution()

--- Part 1: 開始執行 UDN 內文爬取任務 ---
UDN 資料庫中已存在 0 篇文章。
正在從 UDN RSS Feed 檢查新文章...
從 RSS 發現 20 篇新文章。
正在抓取內文: https://udn.com/news/story/124635/9090209...
正在抓取內文: https://udn.com/news/story/7254/9090216...
正在抓取內文: https://udn.com/news/story/7005/9090217...
正在抓取內文: https://udn.com/news/story/124626/9090213...
正在抓取內文: https://udn.com/news/story/7241/9090214...
正在抓取內文: https://udn.com/news/story/6656/9089636...
正在抓取內文: https://udn.com/news/story/7241/9090211...
正在抓取內文: https://udn.com/news/story/6928/9089662...
正在抓取內文: https://udn.com/news/story/7003/9090207...
正在抓取內文: https://udn.com/news/story/7266/9090135...
正在抓取內文: https://udn.com/news/story/7241/9090017...
正在抓取內文: https://udn.com/news/story/7254/9090176...
正在抓取內文: https://udn.com/news/story/124626/9090145...
正在抓取內文: https://udn.com/news/story/7266/9090019...
正在抓取內文: https://udn.com/news/story/7001/9090142...
正在抓取內文: https://udn.com/news/story/6928/9090018...
正在抓取內文: https://udn.com/news/story/7321/9090138...
正在抓取內文: https://udn.com/news/sto